# Challenge 22 — Simple End-to-End Submit Notebook

Fast, reliable CPU pipeline for same-day submission.
This notebook:
1. Loads local files from `data/`
2. Builds minimal features
3. Trains one LightGBM model
4. Generates `submission.csv` (`;`) and `submission_comma.csv` (`,`) 

In [1]:
# Optional one-time installs (uncomment only if needed)
# !pip3 install -U lightgbm pandas scikit-learn

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupShuffleSplit

RANDOM_SEED = 42

BASE_DIR = Path('/Users/raphaelcaillon/Documents/GitHub/challenge_data_ens_22')
TRAIN_INPUT_PATH = BASE_DIR / 'data' / 'training_input.csv'
TEST_INPUT_PATH = BASE_DIR / 'data' / 'testing_input.csv'
TRAIN_TARGET_PATH = BASE_DIR / 'data' / 'challenge_34_cfm_trainingoutputfile (2).csv'

SUBMISSION_SEMICOLON_PATH = BASE_DIR / 'submission.csv'
SUBMISSION_COMMA_PATH = BASE_DIR / 'submission_comma.csv'

LGBM_PARAMS = {
    'objective': 'regression',
    'learning_rate': 0.03,
    'num_leaves': 127,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'min_data_in_leaf': 80,
    'n_estimators': 3000,
    'seed': RANDOM_SEED,
    'metric': 'None',
    'verbosity': -1,
}

print('Train input:', TRAIN_INPUT_PATH)
print('Test input:', TEST_INPUT_PATH)
print('Train target:', TRAIN_TARGET_PATH)

Train input: /Users/raphaelcaillon/Documents/GitHub/challenge_data_ens_22/data/training_input.csv
Test input: /Users/raphaelcaillon/Documents/GitHub/challenge_data_ens_22/data/testing_input.csv
Train target: /Users/raphaelcaillon/Documents/GitHub/challenge_data_ens_22/data/challenge_34_cfm_trainingoutputfile (2).csv


In [3]:
# Robust loading
train_df = pd.read_csv(TRAIN_INPUT_PATH, sep=';')
test_df = pd.read_csv(TEST_INPUT_PATH, sep=';')

target_df = pd.read_csv(TRAIN_TARGET_PATH, sep=';')
if len(target_df.columns) == 1:
    target_df = pd.read_csv(TRAIN_TARGET_PATH, sep=',')

assert 'ID' in target_df.columns and 'TARGET' in target_df.columns, 'Target file must contain ID and TARGET'
assert train_df['ID'].is_unique, 'training_input ID must be unique'
assert test_df['ID'].is_unique, 'testing_input ID must be unique'
assert target_df['ID'].is_unique, 'target ID must be unique'

train_df = train_df.merge(target_df[['ID', 'TARGET']], on='ID', how='inner')

print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)
print('Target shape:', target_df.shape)

Train shape: (636313, 112)
Test shape: (635397, 111)
Target shape: (636313, 2)


In [4]:
# Minimal feature engineering
vol_cols = [c for c in train_df.columns if c.startswith('volatility ')]
ret_cols = [c for c in train_df.columns if c.startswith('return ')]

raw_feature_cols = ['ID', 'date', 'product_id'] + vol_cols + ret_cols


def build_features(df, vol_cols_local, ret_cols_local):
    X = df[raw_feature_cols].copy()
    vol = df[vol_cols_local]
    ret = df[ret_cols_local]

    # Four cheap engineered features
    X['vol_mean'] = vol.mean(axis=1)
    X['vol_std'] = vol.std(axis=1)
    X['ret_abs_mean'] = ret.abs().mean(axis=1)
    X['ret_pos_ratio'] = (ret > 0).mean(axis=1)
    return X


def sanitize_feature_names(columns):
    used = {}
    sanitized = []
    for col in columns:
        clean = ''.join(ch if (ch.isalnum() or ch == '_') else '_' for ch in col)
        if not clean:
            clean = 'f'
        if clean in used:
            used[clean] += 1
            clean = f'{clean}_{used[clean]}'
        else:
            used[clean] = 0
        sanitized.append(clean)
    return sanitized


X_train = build_features(train_df, vol_cols, ret_cols)
X_test = build_features(test_df, vol_cols, ret_cols)
y = train_df['TARGET'].values

sanitized_names = sanitize_feature_names(X_train.columns)
rename_map = dict(zip(X_train.columns, sanitized_names))
X_train = X_train.rename(columns=rename_map)
X_test = X_test.rename(columns=rename_map)

# Cast categorical IDs to category dtype for LightGBM
for frame in (X_train, X_test):
    frame['date'] = frame['date'].astype('category')
    frame['product_id'] = frame['product_id'].astype('category')

print('Feature count:', X_train.shape[1])
print('Vol columns:', len(vol_cols), '| Return columns:', len(ret_cols))

Feature count: 115
Vol columns: 54 | Return columns: 54


In [5]:
# Validation split and holdout MAPE on original target scale

def safe_mape(y_true, y_pred, eps=1e-6):
    y_true = np.maximum(y_true, eps)
    return np.mean(np.abs(y_true - y_pred) / y_true)


def eval_mape_on_original_scale(y_true_log, y_pred_log):
    y_true = np.expm1(y_true_log)
    y_pred = np.expm1(y_pred_log)
    return 'mape', safe_mape(y_true, y_pred), False

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx, valid_idx = next(splitter.split(X_train, y, groups=train_df['date']))

X_tr = X_train.iloc[train_idx]
X_va = X_train.iloc[valid_idx]
y_tr = y[train_idx]
y_va = y[valid_idx]

holdout_model = lgb.LGBMRegressor(**LGBM_PARAMS)
holdout_model.fit(
    X_tr,
    np.log1p(y_tr),
    eval_set=[(X_va, np.log1p(y_va))],
    eval_metric=eval_mape_on_original_scale,
    callbacks=[lgb.early_stopping(200, verbose=False)],
)

va_pred = np.expm1(holdout_model.predict(X_va, num_iteration=holdout_model.best_iteration_))
holdout_mape = safe_mape(y_va, va_pred)
print(f'Holdout MAPE: {holdout_mape:.6f}')
print('Best iteration:', holdout_model.best_iteration_)

Holdout MAPE: 0.244716
Best iteration: 285


In [6]:
# Full training and test prediction
best_iteration = holdout_model.best_iteration_ if holdout_model.best_iteration_ is not None else 300
if best_iteration <= 0:
    best_iteration = 300

final_params = dict(LGBM_PARAMS)
final_params['n_estimators'] = int(best_iteration)

final_model = lgb.LGBMRegressor(**final_params)
final_model.fit(X_train, np.log1p(y))

test_pred = np.expm1(final_model.predict(X_test))
test_pred = np.maximum(test_pred, 1e-6)

submission_df = pd.DataFrame({
    'ID': test_df['ID'].values,
    'TARGET': test_pred,
})

In [7]:
# Save outputs and verify
assert list(submission_df.columns) == ['ID', 'TARGET']
assert len(submission_df) == len(test_df), 'Submission rows must match testing rows'
assert submission_df['TARGET'].notna().all(), 'Submission TARGET contains NaN'
assert (submission_df['TARGET'] > 0).all(), 'Submission TARGET must be strictly positive'

submission_df.to_csv(SUBMISSION_SEMICOLON_PATH, sep=';', index=False)
submission_df.to_csv(SUBMISSION_COMMA_PATH, index=False)

print('Rows:', len(submission_df), '| Expected:', len(test_df))
print('Prediction min/max:', float(submission_df['TARGET'].min()), float(submission_df['TARGET'].max()))
print('First 5 rows:')
print(submission_df.head())
print(f"Saved semicolon file: {SUBMISSION_SEMICOLON_PATH}")
print(f"Saved comma file: {SUBMISSION_COMMA_PATH}")
print('Ready to upload.')

Rows: 635397 | Expected: 635397
Prediction min/max: 0.02426233861630556 2.429526275402249
First 5 rows:
       ID    TARGET
0  636314  0.149877
1  636315  0.100306
2  636316  0.159802
3  636317  0.126837
4  636318  0.126201
Saved semicolon file: /Users/raphaelcaillon/Documents/GitHub/challenge_data_ens_22/submission.csv
Saved comma file: /Users/raphaelcaillon/Documents/GitHub/challenge_data_ens_22/submission_comma.csv
Ready to upload.
